# Fast-fail analysis — AE trivial inversion (latent recovery)

**Question**: can the foundation model's latent space be navigated by gradient
optimization well enough to represent true velocity models? (the
reparameterized-FWI premise)

**Data**: `notebooks/data/` — extracted from MLflow experiment
`encdec.highres.inversion` by `notebooks/data/extract_from_mlflow.py`.
Provenance and column meanings: `notebooks/data/DATA_DICTIONARY.md`.

**Campaign**: 4 refit models (encdec 3×3 seed 7, encdec 5×5 seed 79,
encdec 7×7 seed 571, transencdec 5×5 seed 571) × 4 latent inits ×
5 lrs × 4 epoch budgets; every run inverts the SAME 3 target models —
the hardcoded inverted test indices [1, 10, 20], hand-picked difficult
examples from the refit's test set (whose 100-model MCCV holdout is
logged as context only). 460 runs total → **321 in the clean set**.

Reading order: audit the cleaning (cell 2) → sweep map (3) → final-state
tables (4) → baseline gap (5) → convergence & best-so-far (6) →
difficulty spread (7) → verdict (8).

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("data") if Path("data/runs.csv").exists() else Path("notebooks/data")
runs = pd.read_csv(DATA / "runs.csv", parse_dates=["start_time_iso"])

print("total runs:", len(runs))
display(runs["group"].value_counts().rename("runs").to_frame())

sweep = runs[runs["group"] == "sweep_full_samples"].copy()
kept = sweep[sweep["kept_in_clean_set"] == True].copy()     # noqa: E712
dropped = sweep[sweep["kept_in_clean_set"] == False].copy()  # noqa: E712
print(f"sweep {len(sweep)} = kept {len(kept)} + duplicates {len(dropped)}")
assert len(kept) == kept["combo"].nunique(), "kept must be one row per combo"

print("\nkept runs per refit model:")
display(kept.groupby(["arch_family", "kernel", "seed", "refit_short"])
        .size().rename("kept").to_frame())

In [ ]:
# Sweep map: is the 4x5x4 factorial complete for every refit model?
INITS = ["trained", "zeros", "random", "randomn"]
LRS = [0.001, 0.01, 0.1, 0.5, 1.0]
BUDGETS = [50, 100, 500, 1000]
key_cols = ["refit_short", "latent_init", "lr", "max_epochs"]

present = set(map(tuple, kept[key_cols].itertuples(index=False, name=None)))
expected = {(r, i, float(lr), float(ep))
            for r in kept["refit_short"].unique()
            for i in INITS for lr in LRS for ep in BUDGETS}
print(f"grid cells present: {len(present & expected)} / {len(expected)}")
print("missing:", sorted(expected - present) or "none")

outside = kept[~kept[key_cols].apply(lambda r: tuple(r) in expected, axis=1)]
print("\nruns outside the 4x5x4 grid (exploratory):")
display(outside[["combo", "inversion_mse_mean_test_set", "start_time_iso"]])

print("\nthe duplicate runs (dropped by keep-last) — note the random-init re-runs,")
print("which are distinct latent draws (ADR-0002 J-draws material):")
display(dropped[["combo", "latent_init", "lr", "max_epochs", "start_time_iso"]]
        .sort_values(["latent_init", "lr", "max_epochs", "start_time_iso"]))

In [ ]:
# Final-state tables. CAVEAT: 'final' = state at schedule end; with
# cosine_annealing_wr that can be a peak-lr point (quantified in the next cells).
mse = "inversion_mse_mean_test_set"
ssim = "inversion_ssim_mean_test_set"

for ep in [1000, 500, 100, 50]:
    sub = kept[kept["max_epochs"] == ep]
    t = sub.pivot_table(index="latent_init", columns="lr", values=mse, aggfunc="mean")
    print(f"\n=== final MSE @ {ep} epochs — mean over the 4 refit models "
          f"(each run = mean over its 3 inverted targets, the difficult "
          f"examples [1, 10, 20]) ===")
    display(t.round(4))

print("\n=== per-init best across all 20 (lr, budget) combos ===")
per_combo = kept.groupby(["latent_init", "lr", "max_epochs"])[mse].mean().reset_index()
display(per_combo.loc[per_combo.groupby("latent_init")[mse].idxmin()].round(4))

print("\n=== per refit model, clean grid (all budgets) ===")
display(kept.groupby(["arch_family", "kernel", "seed"])[mse]
        .agg(["count", "mean", "min", "max"]).round(4))

t = kept[kept["max_epochs"] == 1000].pivot_table(index="latent_init", columns="lr",
                                                 values=ssim, aggfunc="mean")
print("\n=== final SSIM @ 1000 epochs ===")
display(t.round(3))

In [ ]:
# Baseline gap: refit prediction (mean over its 100-model holdout) vs trivial
# inversion (mean over the 3 hand-picked DIFFICULT targets [1, 10, 20]).
# Difficulty mismatch inflates the gap; cell 6 (best-so-far) quantifies the
# final-state artifact on top of that.
base = pd.read_csv(DATA / "refit_baselines.csv").set_index("refit_short")

inv = kept[kept["max_epochs"] == 1000].pivot_table(
    index="refit_short", columns="latent_init", values=mse, aggfunc="mean")
cmp = pd.DataFrame({
    "refit_mse": base["mse_mean_test_set"],
    "inv@1000_mean_over_inits": inv.mean(axis=1),
    "inv@1000_best_init": inv.min(axis=1),
})
cmp["ratio_mean"] = cmp["inv@1000_mean_over_inits"] / cmp["refit_mse"]
cmp["refit_ssim"] = base["ssim_mean_test_set"]
inv_ssim = kept[kept["max_epochs"] == 1000].pivot_table(
    index="refit_short", columns="latent_init", values=ssim, aggfunc="mean")
cmp["inv_ssim@1000_mean"] = inv_ssim.mean(axis=1)

print("=== refit prediction vs trivial inversion ===")
display(cmp.round(4))

In [ ]:
# Convergence + best-so-far vs final (quantifies the warm-restart artifact).
import matplotlib.pyplot as plt

hist = pd.read_csv(DATA / "train_loss_history.csv")
print("history rows:", len(hist))

# clean curve: last logged value per step (steps may repeat)
curves = (hist.sort_values("step")
              .groupby(["run_id", "step"], as_index=False)["value"].last())
curves = curves.merge(kept[["run_id", "combo", "refit_short", "latent_init",
                            "lr", "max_epochs"]], on="run_id")

KEY = [("trained", 0.01), ("trained", 0.1), ("random", 0.01), ("random", 0.1)]
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (init, lr) in zip(axes.ravel(), KEY):
    sel = curves[(curves["latent_init"] == init) & (curves["lr"] == lr)
                 & (curves["max_epochs"] == 1000)]
    for refit, g in sel.groupby("refit_short"):
        ax.plot(g["step"], g["value"], lw=0.8, label=refit)
    ax.set_yscale("log")
    ax.set_title(f"init={init}, lr={lr}, 1000 epochs")
    ax.set_xlabel("step")
    ax.set_ylabel("train_loss")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

bsf = curves.groupby("run_id")["value"].min().rename("best_so_far")
bf = pd.concat([bsf, kept.set_index("run_id")[mse].rename("final_mse")],
               axis=1).dropna()
bf["final/best"] = bf["final_mse"] / bf["best_so_far"]
print("=== final-state penalty over all 321 clean runs ===")
display(bf.describe().round(4))
print("\nworst final/best cases:")
display(bf.sort_values("final/best", ascending=False).head(10).round(4))
print("\nbest-so-far floor vs refit baseline (~0.0009):", bf["best_so_far"].min())

In [ ]:
# Per-model difficulty spread (comment i): MSE varies a lot across test models,
# so single-model readings must be difficulty-matched. Here the spread is over
# the 3 inverted targets ([1, 10, 20]) — a small sample, but it brackets the
# difficulty range those specific models span.
spread = kept[["combo", "latent_init", "lr", "max_epochs",
               "inversion_mse_min_test_set", mse,
               "inversion_mse_max_test_set", "inversion_mse_std_test_set"]].copy()
spread["max/min"] = spread["inversion_mse_max_test_set"] / spread["inversion_mse_min_test_set"]

print("per-run spread of per-model MSE over the 3 inverted targets ([1, 10, 20]):")
display(spread[["max/min", "inversion_mse_std_test_set"]].describe().round(3))

refit_spread = (base["mse_max_test_set"] / base["mse_min_test_set"]).rename("refit max/min")
print("\nrefit baseline spread on the same holdouts:")
display(refit_spread.round(2).to_frame())

ax = spread["max/min"].plot(kind="hist", bins=40, figsize=(7, 3),
                            title="per-run max/min of per-model MSE")
ax.set_xlabel("max/min")
plt.show()

## Verdict — *draft after exploring*

**Findings**
- (fill after running the cells — the numbers you trust are the ones you produced)

**Caveats**
- final-state reporting at warm-restart peaks → quantified in the convergence cell; fix lands in the final experiments
- per-model difficulty variance → spread cell; single-model readings must be difficulty-matched

**Next (post-refactor AE+VAE rerun checklist)**
- best-so-far / cold-end reporting
- `init=encoder-of-target` — measures the manifold-coverage ceiling
- per-model paired analysis
- 10-seed MCCV (ADR-0002)